
---

# 🟩 PARTE 1 — SQL aplicado a negocio (45 min)

Ya no estás practicando consultas “porque sí”.
Ahora estás respondiendo preguntas como haría un analista en una empresa.

Tu dataset es educativo, pero lo tratarás como si fuera de clientes.

---

## 🎯 Objetivo de hoy

Segmentar estudiantes y comparar rendimiento promedio.

Piensa así:

> “Dividamos estudiantes en grupos según hábitos y veamos quién rinde mejor”.

Eso es **segmentación de negocio**.







In [ ]:
import duckdb
import os


In [21]:
os.getcwd()
os.listdir()
os.listdir('../data/student_performance')

['student_performance.csv']

In [ ]:

con = duckdb.connect()


'/home/andres/Escritorio/Bootocamp personal/career-2026/notebooks'

In [22]:
con.execute("""
CREATE OR REPLACE TABLE students AS
SELECT *
FROM read_csv_auto('../data/student_performance/student_performance.csv')
""")

In [23]:
con.execute("DESCRIBE students").fetchdf()

,column_name,column_type,null,key,default,extra
0,Hours_Studied,BIGINT,YES,None,None,None
1,Attendance,BIGINT,YES,None,None,None
2,Parental_Involvement,VARCHAR,YES,None,None,None
3,Access_to_Resources,VARCHAR,YES,None,None,None
4,Extracurricular_Activities,BOOLEAN,YES,None,None,None
5,Sleep_Hours,BIGINT,YES,None,None,None
6,Previous_Scores,BIGINT,YES,None,None,None
7,Motivation_Level,VARCHAR,YES,None,None,None
8,Internet_Access,BOOLEAN,YES,None,None,None
9,Tutoring_Sessions,BIGINT,YES,None,None,None


---

## 🧩 Ejercicio 1 — Segmentación por horas de estudio

Divide estudiantes en niveles:

* Bajo estudio
* Medio estudio
* Alto estudio

Y calcula:

> Promedio de `Exam_Score` por grupo

💡 Estás simulando algo como:
“Clientes que compran poco / medio / mucho”.

---

In [32]:
con.execute("""
SELECT Hours_Studied,
        case 
            when Hours_Studied < 10 then 'Low'
            when Hours_Studied < 20 then 'Medium'
            else 'High'

        end as Study_Level
FROM students

            """).fetchdf()

,Hours_Studied,Study_Level
0,23,High
1,19,Medium
2,24,High
3,29,High
4,19,Medium
...,...,...
6602,25,High
6603,23,High
6604,20,High
6605,10,Medium


In [35]:
con.execute("""
SELECT 
        case 
            when Hours_Studied < 10 then 'Low'
            when Hours_Studied < 20 then 'Medium'
            else 'High'

        end as Study_Level, avg(Exam_Score) as Average_Score
FROM students
group by Study_Level

            """).fetchdf()

,Study_Level,Average_Score
0,High,68.489512
1,Low,63.819188
2,Medium,65.990028


## 🧩 Ejercicio 2 — Segmentación por asistencia

Agrupa por niveles de `Attendance`:

* Baja asistencia
* Media asistencia
* Alta asistencia

Luego compara:

> Promedio de notas por grupo

---


In [40]:
con.execute("""
SELECT 
        case 
            when Attendance < 60 then 'Low'
            when Attendance < 85 then 'Medium'
            else 'High'
        end as Attendance_Level, avg(Exam_Score) as Average_Score
FROM students
group by Attendance_Level

            """).fetchdf()

,Attendance_Level,Average_Score
0,High,69.694544
1,Medium,65.728271


## 🧩 Ejercicio 3 — Segmentación combinada (nivel pro)

Cruza:

* Nivel de estudio
* Nivel de asistencia

Y calcula:

> Promedio de rendimiento por combinación

Esto responde preguntas tipo:

> “¿Qué importa más: estudiar mucho o asistir mucho?”

---


In [44]:
con.execute("""
SELECT
        case 
            when Hours_Studied < 10 then 'Low'
            when Hours_Studied < 20 then 'Medium'
            else 'High'

        end as Study_Level,
        case 
            when Attendance < 60 then 'Low'
            when Attendance < 85 then 'Medium'
            else 'High'
        end as Attendance_Level,
        avg(Exam_Score) as Average_Score
FROM students
group by Study_Level, Attendance_Level
order by average_score desc
          
        """).fetchdf()

,Study_Level,Attendance_Level,Average_Score
0,High,High,70.964232
1,Medium,High,68.433803
2,High,Medium,66.970265
3,Low,High,66.221154
4,Medium,Medium,64.496845
5,Low,Medium,62.323353
